In [1]:
import os
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time
import math
from scipy.stats import kendalltau, spearmanr

# ============ 配置 ============
client = OpenAI(
    base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
    api_key='ida_TbtBzRyj8HeV7kmxkeRsgImHnYelRIUMQ3vW9VIf'
)

MODEL = 'qwen-2.5-72b-instruct'

# ============ Zero-Shot Prompt ============
ZERO_SHOT_PROMPT_TEMPLATE = """You are an information retrieval expert. Analyze whether Pseudo-Relevance Feedback (PRF) will improve or hurt retrieval quality for this query.

Query: "{query}"

Top-5 Retrieved Documents:
{documents}

Consider:
1. Query clarity and specificity
2. Quality and relevance of top documents
3. Topic consistency across documents
4. Risk of query drift with PRF

Answer with ONLY one word: "Benefit" or "Hurt"
"""

# ============ 加载数据 ============
print("=" * 80)
print("Zero-Shot PRF Prediction with Qwen-2.5-72B")
print("=" * 80)

print("\n1. 加载数据...")
df_preference = pd.read_csv('preference.csv')
df_queries = pd.read_csv('result_with_text.csv')
df_colbert = pd.read_csv('df_colbert_deduped.csv')

# 提取 query text 映射
if 'query_text' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query_text'].first().to_dict()
elif 'query' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query'].first().to_dict()
else:
    print("错误: 找不到 query 或 query_text 列")
    exit(1)

# 过滤有效查询 (b_preference_ratio > 0)
eval_df = df_preference[df_preference['b_preference_ratio'] > 0].copy()
print(f"✓ 加载了 {len(eval_df)} 个有效查询 (b_preference_ratio > 0)")

# ============ 预测函数 ============
def predict_with_logprobs(qid, query, top5_docs, verbose=False):
    """使用 logprobs 获取真实置信度"""
    
    # 格式化文档
    docs_text = "\n".join([
        f"[{i+1}] [Score: {row['score']:.2f}] {str(row['passage_text'])[:200]}..."
        for i, (_, row) in enumerate(top5_docs.iterrows())
    ])
    
    # 构建 prompt
    prompt = ZERO_SHOT_PROMPT_TEMPLATE.format(
        query=query,
        documents=docs_text
    )
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are an expert in information retrieval. Be concise."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=5,
            logprobs=True,
            top_logprobs=10
        )
        
        # 提取预测
        prediction_text = response.choices[0].message.content.strip()
        
        # 提取 logprobs
        if not response.choices[0].logprobs or not response.choices[0].logprobs.content:
            if verbose:
                print(f"  ⚠️ No logprobs returned")
            return {
                "prediction": prediction_text,
                "confidence": 0.5,
                "raw_probs": {},
                "success": False
            }
        
        first_token_logprobs = response.choices[0].logprobs.content[0].top_logprobs
        
        # 转换为概率
        raw_probs = {}
        for item in first_token_logprobs:
            token = item.token.strip().lower()
            prob = math.exp(item.logprob)
            raw_probs[token] = prob
        
        # 提取 Benefit 和 Hurt 概率
        benefit_prob = sum(prob for token, prob in raw_probs.items() if 'benefit' in token)
        hurt_prob = sum(prob for token, prob in raw_probs.items() if 'hurt' in token)
        
        # 归一化
        total = benefit_prob + hurt_prob
        if total > 0:
            confidence = benefit_prob / total
        else:
            # Fallback: 从预测文本推断
            if 'benefit' in prediction_text.lower():
                confidence = 0.9
            elif 'hurt' in prediction_text.lower():
                confidence = 0.1
            else:
                confidence = 0.5
        
        final_prediction = "Benefit" if confidence > 0.5 else "Hurt"
        
        if verbose:
            print(f"  Prediction: {final_prediction}")
            print(f"  Confidence: {confidence:.3f}")
            print(f"  Raw probs: {raw_probs}")
        
        return {
            "prediction": final_prediction,
            "confidence": confidence,
            "raw_probs": raw_probs,
            "raw_text": prediction_text,
            "success": True
        }
        
    except Exception as e:
        if verbose:
            print(f"  ✗ Error: {e}")
        return {
            "prediction": "Hurt",
            "confidence": 0.5,
            "raw_probs": {},
            "error": str(e),
            "success": False
        }

# ============ 批量预测 ============
print("\n2. 运行预测...")
print("-" * 80)

results = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Processing queries"):
    qid = row['qid']
    query = qid_to_query.get(qid, f"[Unknown query for QID {qid}]")
    
    # 获取 top-5 文档
    top5 = df_colbert[df_colbert['qid'] == qid].nlargest(5, 'score')
    
    if len(top5) == 0:
        print(f"  ⚠️ QID {qid}: 没有检索结果")
        continue
    
    # 预测
    result = predict_with_logprobs(qid, query, top5, verbose=False)
    
    # 保存结果
    results.append({
        'qid': qid,
        'query': query,
        'true_b_pref_ratio': row['b_preference_ratio'],
        'true_preference': row['preference'],
        'llm_prediction': result['prediction'],
        'llm_confidence': result['confidence'],
        'raw_text': result.get('raw_text', ''),
        'success': result['success']
    })
    
    # 限速
    time.sleep(0.5)

results_df = pd.DataFrame(results)

# 保存原始结果
results_df.to_csv('qwen_zero_shot_predictions.csv', index=False, encoding='utf-8')
print(f"\n✓ 预测完成: {len(results_df)} 个查询")
print(f"✓ 成功: {results_df['success'].sum()}/{len(results_df)}")

# ============ 评估 ============
print("\n" + "=" * 80)
print("3. EVALUATION RESULTS")
print("=" * 80)

# 只评估成功的预测
eval_results = results_df[results_df['success'] == True].copy()
print(f"\n有效样本数: {len(eval_results)}")

if len(eval_results) < 10:
    print("⚠️ 有效样本太少，无法进行可靠评估")
    exit(1)

# ========== 相关性分析 ==========
print("\n" + "-" * 80)
print("A. CORRELATION ANALYSIS")
print("-" * 80)

tau, p_tau = kendalltau(eval_results['llm_confidence'], 
                        eval_results['true_b_pref_ratio'])
rho, p_rho = spearmanr(eval_results['llm_confidence'], 
                       eval_results['true_b_pref_ratio'])

print(f"\nKendall's Tau:  τ = {tau:.4f} (p = {p_tau:.4f})")
print(f"Spearman's Rho: ρ = {rho:.4f} (p = {p_rho:.4f})")

# 显著性
if p_tau < 0.001:
    sig = "*** (p<0.001)"
elif p_tau < 0.01:
    sig = "** (p<0.01)"
elif p_tau < 0.05:
    sig = "* (p<0.05)"
else:
    sig = "(not significant)"

print(f"Significance: {sig}")

# 相关性强度
if abs(tau) < 0.1:
    strength = "Negligible"
elif abs(tau) < 0.3:
    strength = "Weak"
elif abs(tau) < 0.5:
    strength = "Moderate"
elif abs(tau) < 0.7:
    strength = "Strong"
else:
    strength = "Very Strong"

direction = "Positive" if tau > 0 else "Negative"
print(f"Correlation: {strength} {direction}")

# ========== 分类性能 ==========
print("\n" + "-" * 80)
print("B. CLASSIFICATION PERFORMANCE")
print("-" * 80)

# 定义 ground truth label (threshold = 0.55)
eval_results['true_label'] = eval_results['true_b_pref_ratio'].apply(
    lambda x: 'Benefit' if x > 0.55 else ('Hurt' if x < 0.45 else 'Neutral')
)

# 排除 Neutral
clf_df = eval_results[eval_results['true_label'] != 'Neutral'].copy()

if len(clf_df) > 0:
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    
    accuracy = accuracy_score(clf_df['true_label'], clf_df['llm_prediction'])
    f1 = f1_score(clf_df['true_label'], clf_df['llm_prediction'], 
                  pos_label='Benefit', zero_division=0)
    precision = precision_score(clf_df['true_label'], clf_df['llm_prediction'], 
                                pos_label='Benefit', zero_division=0)
    recall = recall_score(clf_df['true_label'], clf_df['llm_prediction'], 
                         pos_label='Benefit', zero_division=0)
    
    print(f"\nSample Size:    {len(clf_df)}")
    print(f"  PRF-Benefit:  {(clf_df['true_label'] == 'Benefit').sum()}")
    print(f"  PRF-Hurt:     {(clf_df['true_label'] == 'Hurt').sum()}")
    print(f"\nAccuracy:       {accuracy:.4f}")
    print(f"F1-Score:       {f1:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall:         {recall:.4f}")

# ========== 对比 QPP Baselines ==========
print("\n" + "-" * 80)
print("C. COMPARISON WITH QPP BASELINES")
print("-" * 80)

print(f"\n{'Method':<15} {'Kendall τ':<12} {'P-value':<12} {'Strength':<20}")
print("-" * 80)
print(f"{'NQC':<15} {-0.1054:<12.4f} {0.3392:<12.4f} {'Weak Negative':<20}")
print(f"{'WIG':<15} {0.0051:<12.4f} {0.9628:<12.4f} {'Negligible':<20}")
print(f"{'UEF':<15} {0.0057:<12.4f} {0.9654:<12.4f} {'Negligible':<20}")
print(f"{'SMV':<15} {-0.0463:<12.4f} {0.6748:<12.4f} {'Negligible':<20}")
print(f"{'DENSE_QPP':<15} {0.1260:<12.4f} {0.2534:<12.4f} {'Weak Positive':<20}")
print("-" * 80)
print(f"{'Qwen-2.5-72B':<15} {tau:<12.4f} {p_tau:<12.4f} {strength + ' ' + direction:<20} {sig}")
print("-" * 80)

# 计算提升
best_qpp_tau = 0.126  # DENSE_QPP
if abs(tau) > abs(best_qpp_tau):
    improvement = ((abs(tau) - abs(best_qpp_tau)) / abs(best_qpp_tau)) * 100
    print(f"\n✅ Relative improvement over best QPP: {improvement:.1f}%")
    print(f"   (|τ| = {abs(tau):.4f} vs {abs(best_qpp_tau):.4f})")

# ========== 案例分析 ==========
print("\n" + "-" * 80)
print("D. SAMPLE PREDICTIONS")
print("-" * 80)

# 选择一些有代表性的案例
print("\n高置信度 Benefit 预测:")
high_benefit = eval_results[eval_results['llm_confidence'] > 0.8].head(3)
for _, row in high_benefit.iterrows():
    print(f"  QID {row['qid']}: {row['query'][:60]}...")
    print(f"    LLM: {row['llm_prediction']} ({row['llm_confidence']:.3f})")
    print(f"    True: {row['true_preference']} (ratio={row['true_b_pref_ratio']:.3f})")

print("\n高置信度 Hurt 预测:")
high_hurt = eval_results[eval_results['llm_confidence'] < 0.2].head(3)
for _, row in high_hurt.iterrows():
    print(f"  QID {row['qid']}: {row['query'][:60]}...")
    print(f"    LLM: {row['llm_prediction']} ({row['llm_confidence']:.3f})")
    print(f"    True: {row['true_preference']} (ratio={row['true_b_pref_ratio']:.3f})")

print("\n不确定案例 (接近0.5):")
uncertain = eval_results[(eval_results['llm_confidence'] > 0.45) & 
                         (eval_results['llm_confidence'] < 0.55)].head(3)
for _, row in uncertain.iterrows():
    print(f"  QID {row['qid']}: {row['query'][:60]}...")
    print(f"    LLM: {row['llm_prediction']} ({row['llm_confidence']:.3f})")
    print(f"    True: {row['true_preference']} (ratio={row['true_b_pref_ratio']:.3f})")

# ========== 总结 ==========
print("\n" + "=" * 80)
print("4. SUMMARY")
print("=" * 80)

print(f"""
🎯 Zero-Shot Performance:
   - Sample Size: {len(eval_results)}
   - Kendall's τ: {tau:.4f} ({strength} {direction})
   - P-value: {p_tau:.4f} {sig}
   - Spearman's ρ: {rho:.4f}
""")

if len(clf_df) > 0:
    print(f"""   - F1-Score: {f1:.4f}
   - Accuracy: {accuracy:.4f}
   - Precision: {precision:.4f}
   - Recall: {recall:.4f}
""")

print("\n📊 Comparison with QPP:")
print(f"   - Best QPP (DENSE): τ = 0.126 (not significant)")
print(f"   - Qwen-2.5-72B: τ = {tau:.4f} {sig}")

if abs(tau) > 0.126 and p_tau < 0.05:
    print("\n✅ SUCCESS: Significantly outperforms all QPP baselines!")
elif abs(tau) > 0.05 and p_tau < 0.05:
    print("\n✅ GOOD: Shows significant correlation (better than QPP)")
elif abs(tau) > 0.126:
    print("\n⚠️ PROMISING: Higher correlation but lacks significance (need more data)")
else:
    print("\n⚠️ NEEDS IMPROVEMENT: Consider few-shot prompting")

print("\n📁 Output Files:")
print("   ✓ qwen_zero_shot_predictions.csv - Full predictions")

print("\n" + "=" * 80)
print("Analysis complete!")
print("=" * 80)

Zero-Shot PRF Prediction with Qwen-2.5-72B

1. 加载数据...
✓ 加载了 40 个有效查询 (b_preference_ratio > 0)

2. 运行预测...
--------------------------------------------------------------------------------


Processing queries: 100%|██████████| 40/40 [00:51<00:00,  1.28s/it]


✓ 预测完成: 40 个查询
✓ 成功: 40/40

3. EVALUATION RESULTS

有效样本数: 40

--------------------------------------------------------------------------------
A. CORRELATION ANALYSIS
--------------------------------------------------------------------------------

Kendall's Tau:  τ = -0.2127 (p = 0.1089)
Spearman's Rho: ρ = -0.2567 (p = 0.1098)
Significance: (not significant)
Correlation: Weak Negative

--------------------------------------------------------------------------------
B. CLASSIFICATION PERFORMANCE
--------------------------------------------------------------------------------

Sample Size:    26
  PRF-Benefit:  13
  PRF-Hurt:     13

Accuracy:       0.5000
F1-Score:       0.6667
Precision:      0.5000
Recall:         1.0000

--------------------------------------------------------------------------------
C. COMPARISON WITH QPP BASELINES
--------------------------------------------------------------------------------

Method          Kendall τ    P-value      Strength            
-----